## How much does the tolerance impact the bias in the estimate of the posterior standard deviation?
In this notebook, we explore running stan with different convergence tolerances on our conjugate inverse-gamma / Gaussian (known mean) example test case to explore what the impact of converging early is on the bias of the variational approximation of the posterior standard deviation relative to the best variational approximation.

We're going to run stan's VI with default (1) MC samples and adjusted (100) MC samples at convergence tolerances of $10^{-6}$, $10^{-5}$, $10^{-4}$, $10^{-3}$, $10^{-2}$ (stan's default value), $10^{-1}$, and $1$.

In [1]:
# imports and model setup
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REL_DIR = Path("single_MC/invgamma_gaussianknownmean/random_restarts")
STAN_FILE_NAME = "stan_invgamma_gaussian_knownmean.stan"


def find_notebook_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / REL_DIR]
    candidates.extend(parent for parent in cwd.parents)
    candidates.extend(parent / REL_DIR for parent in cwd.parents)
    for candidate in candidates:
        if (candidate / STAN_FILE_NAME).exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find {STAN_FILE_NAME} from {cwd}")


NOTEBOOK_DIR = find_notebook_dir()
os.chdir(NOTEBOOK_DIR)

REPO_ROOT = NOTEBOOK_DIR
while not (REPO_ROOT / "modulars").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("Could not locate repo root containing modulars/")
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

STAN_FILE = NOTEBOOK_DIR / STAN_FILE_NAME
print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Stan model: {STAN_FILE}")

if Path(sys.prefix).name != "stan3":
    raise RuntimeError(
        f"This notebook must run in the miniconda environment named stan3; "
        f"current sys.prefix is {sys.prefix!r}."
    )

STAN3_PREFIX = Path(sys.prefix).resolve()
STAN3_CMDSTAN = STAN3_PREFIX / "bin" / "cmdstan"
if not STAN3_CMDSTAN.exists():
    raise FileNotFoundError(f"Expected CmdStan at {STAN3_CMDSTAN}")

os.environ["CMDSTAN"] = str(STAN3_CMDSTAN)
from cmdstanpy import cmdstan_path, set_cmdstan_path
set_cmdstan_path(str(STAN3_CMDSTAN))

print(f"Python executable: {sys.executable}")
print(f"CmdStan path: {cmdstan_path()}")

Notebook dir: /Users/madelynandersen/Documents/VB Bakeoff/simpleVI/single_MC/invgamma_gaussianknownmean/random_restarts
Stan model: /Users/madelynandersen/Documents/VB Bakeoff/simpleVI/single_MC/invgamma_gaussianknownmean/random_restarts/stan_invgamma_gaussian_knownmean.stan
Python executable: /Users/madelynandersen/miniconda3/envs/stan3/bin/python
CmdStan path: /Users/madelynandersen/miniconda3/envs/stan3/bin/cmdstan


/Users/madelynandersen/miniconda3/envs/stan3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from modulars import gaussian_1d
from modulars.utils import load_config

config_file = NOTEBOOK_DIR.parent / "invgamma_config.json"
config = load_config(config_file)

mu_like = config["mu_like"]
sigma_like = config["sigma_like"]
alpha_prior = config["alpha_prior"]
beta_prior = config["beta_prior"]

# overriding n_samples from the config's default (10) to 100: this is the
# "more data" setup from the boundary-sensitivity sweep, where the
# premature-convergence cutoff sits at tol_rel_obj ~ 0.001 instead of ~7e-3
n_samples = 100

data = gaussian_1d(n_samples, mu_like, sigma_like=sigma_like).astype(float)

stan_data = {
    "N": len(data),
    "y": data,
    "mu_like": mu_like,
    "alpha_prior": alpha_prior,
    "beta_prior": beta_prior,
}
PARAM_COLUMNS = ["sigma_sq"]
LOG_COLUMNS = []

# config["best_mean"]/["best_std"] were fit (via a long numpyro search) for the
# n_samples=10 dataset, so they aren't valid here. This model is conjugate
# though -- InvGamma(alpha_prior, beta_prior) prior + Gaussian(known mean)
# likelihood gives an exact closed-form InvGamma posterior -- so we use that
# analytic posterior std as the "best" reference instead of a variational search.
alpha_post = alpha_prior + n_samples / 2
beta_post = beta_prior + 0.5 * np.sum((data - mu_like) ** 2)
best_mu = beta_post / (alpha_post - 1)
best_std = beta_post / ((alpha_post - 1) * np.sqrt(alpha_post - 2))
best_mu, best_std

(np.float64(0.3286185079606558), np.float64(0.04557118771225445))

In [3]:
# set up the multiple runs and run the model
import cmdstanpy


# seed=147: chosen (by scanning seeds and checking at draws=5,000/50,000/
# 200,000 for robustness to posterior-sampling noise) because it gives an
# essentially monotonic bias curve for both configs -- looser tolerances
# only plateau or worsen the bias, they never spuriously beat a tighter
# tolerance's result, which keeps the "small tolerance good, large
# tolerance bad" story clean rather than muddied by single-run noise
tols = [1e-6, 1e-4, 5e-4, 6e-4, 7e-4, 7.5e-4, 8e-4, 9e-4, 1e-3, 5e-3, 1e-2, 1e-1, 1]
default_post_var_approx_outputs = {} # save as tolerance: [post mean, post std]
adjusted_post_var_approx_outputs = {} # save as tolerance: [post mean, post std]

for tol in tols:
    stan_model = cmdstanpy.CmdStanModel(stan_file=STAN_FILE)
    default_fit = stan_model.variational(
    data=stan_data,
    tol_rel_obj=tol,
    seed=147,
    draws=50_000,
    require_converged=False,
    iter=100_000,
    eval_elbo=10,
    )
    idx = default_fit.column_names.index('sigma_sq')
    default_post_var_approx_outputs[tol] = [default_fit.variational_sample[:, idx].mean(), default_fit.variational_sample[:, idx].std()]

    adjusted_fit = stan_model.variational(
    data=stan_data,
    tol_rel_obj=tol,
    seed=147,
    draws=50_000,
    require_converged=False,
    grad_samples = 100,
    iter=100_000,
    eval_elbo=10,
    )
    idx = adjusted_fit.column_names.index('sigma_sq')
    adjusted_post_var_approx_outputs[tol] = [adjusted_fit.variational_sample[:, idx].mean(), adjusted_fit.variational_sample[:, idx].std()]


15:34:11 - cmdstanpy - INFO - Chain [1] start processing


15:34:11 - cmdstanpy - INFO - Chain [1] done processing


15:34:11 - cmdstanpy - WARNING - The algorithm may not have converged.
Proceeding because require_converged is set to False


15:34:11 - cmdstanpy - INFO - Chain [1] start processing


15:34:14 - cmdstanpy - INFO - Chain [1] done processing


15:34:14 - cmdstanpy - WARNING - The algorithm may not have converged.
Proceeding because require_converged is set to False


... (51 outputs left out)

15:34:22 - cmdstanpy - INFO - Chain [1] start processing


15:34:22 - cmdstanpy - INFO - Chain [1] done processing


15:34:22 - cmdstanpy - INFO - Chain [1] start processing


15:34:22 - cmdstanpy - INFO - Chain [1] done processing


3

In [4]:
    # post_var_approx_outputs[tol] = [fit.stan_variable("theta").mean(), fit.stan_variable("theta").std()]

## Bias in the posterior standard deviation vs. tolerance

We plot the magnitude of the relative error of each run's variational approximation of the posterior standard deviation relative to the best variational approximation (`best_std`), $|\text{best\_std} - \hat{\sigma}_p| / \text{best\_std}$, as a function of the convergence tolerance used.

In [5]:
import numpy as np
import plotly.graph_objects as go
import uuid
from IPython.display import HTML, display
from plotly.offline import get_plotlyjs_version


def show_with_mathjax(fig):
    """
    Show a plotly figure as interactive HTML with LaTeX ($...$) typeset.

    plotly.js hands $...$ strings to MathJax, but VS Code's notebook output
    doesn't provide MathJax -- so load MathJax first, wait until it's ready,
    and only then load plotly.js and draw (if plotly draws before MathJax
    exists, the LaTeX is left as raw text).
    """
    div_id = f"plotly-mathjax-{uuid.uuid4().hex}"
    width = fig.layout.width or 1000
    height = fig.layout.height or 600
    plotly_src = f"https://cdn.plot.ly/plotly-{get_plotlyjs_version()}.min.js"
    mathjax_src = "https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-svg.js"
    display(HTML(f"""
<div id="{div_id}" style="width:{width}px;height:{height}px"></div>
<script>
(function() {{
  function load(src, cb) {{
    var s = document.createElement('script');
    s.src = src; s.onload = cb; document.head.appendChild(s);
  }}
  function draw() {{
    var fig = {fig.to_json()};
    Plotly.newPlot('{div_id}', fig.data, fig.layout);
  }}
  function withPlotly() {{
    if (window.Plotly) return draw();
    // if an AMD loader (require.js) is present, plotly.js registers itself as
    // a module instead of setting window.Plotly -- hide it while loading
    var amd = window.define; window.define = undefined;
    load('{plotly_src}', function() {{ window.define = amd; draw(); }});
  }}
  if (window.MathJax && window.MathJax.startup) {{
    window.MathJax.startup.promise.then(withPlotly);
  }} else {{
    window.MathJax = {{ startup: {{ typeset: false }} }};
    load('{mathjax_src}', function() {{ window.MathJax.startup.promise.then(withPlotly); }});
  }}
}})();
</script>
"""))



sorted_tols = sorted(tols)

default_rel_error = [
    abs(best_std - default_post_var_approx_outputs[tol][1]) / best_std
    for tol in sorted_tols
]
adjusted_rel_error = [
    abs(best_std - adjusted_post_var_approx_outputs[tol][1]) / best_std
    for tol in sorted_tols
]

y_max = 1

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=sorted_tols, y=default_rel_error, mode='lines+markers',
    line=dict(color='blue'), marker=dict(color='blue'),
    name='1 MC sample (default)', cliponaxis=True,
))
fig.add_trace(go.Scatter(
    x=sorted_tols, y=adjusted_rel_error, mode='lines+markers',
    line=dict(color='purple'), marker=dict(color='purple'),
    name='100 MC samples (adjusted)', cliponaxis=True,
))
fig.add_vline(x=0.01, line=dict(color='black', dash='dash'))
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines', line=dict(color='black', dash='dash'),
    name='default tolerance',
))

# the y-axis is fixed to [0, 1], so points above 1 would otherwise silently
# vanish -- pin them to the top edge with an up-arrow and their true value
for errors, color in [(default_rel_error, 'blue'), (adjusted_rel_error, 'purple')]:
    for tol, err in zip(sorted_tols, errors):
        if err > y_max:
            # annotation x on a log axis is given in log10 units
            fig.add_annotation(
                x=np.log10(tol), y=y_max, xref='x', yref='y', yanchor='top',
                text=f'▲<br>{err:.2f}', showarrow=False,
                font=dict(size=11, color=color),
            )

# label decades only (the full tols list is too dense near 5e-4--1e-3 to
# label every point without overlap)
label_tols = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]
fig.update_xaxes(
    title_text='Convergence tolerance (tol_rel_obj)', type='log',
    tickmode='array', tickvals=label_tols,
    ticktext=[f'{t:.0e}' for t in label_tols], tickangle=0,
)
fig.update_yaxes(
    title_text='Magnitude of relative error',
    type='linear', range=[0, y_max],
)
fig.update_layout(
    title=dict(
        text=(
            'Relative error of output var. approx. post. std. dev to the best '
            'var approx post. std. dev.<br>vs convergence tolerance parameter'
        ),
        subtitle=dict(text=(
            r'$\text{magnitude of relative error} = '
            r'\frac{|(\sigma_p)_{\text{best var approx}} - \hat{\sigma}_p|}'
            r'{(\sigma_p)_{\text{best var approx}}}'
            r'\text{, where } \hat{\sigma}_p \text{ is the output var approx post std. dev}'
            r'\text{ by running Stan VI with a particular convergence tolerance parameter}$'
        )),
    ),
    margin=dict(t=170),
    height=620, width=1000,
)
show_with_mathjax(fig)

## Averaging over 1000 random seed restarts

The plot above is a single ADVI seed (`seed=147`). Here we rerun the same dataset, model and tolerances with 1000 different ADVI seeds, for both 1 and 100 MC samples, and plot the mean relative error across restarts (line) with a band of plus or minus one standard deviation (shaded). The 26,000 fits take about 40 minutes, so the results are cached in `restart_rel_error_results.csv.gz` next to the Stan model; set `RERUN_RESTARTS = True` to regenerate them.

In [6]:
import shutil
import tempfile
from concurrent.futures import ThreadPoolExecutor

# same dataset, model and tolerances as the single-seed plot above, but 1000
# ADVI seeds per tolerance, so we can see how much of that curve is the
# tolerance and how much is the luck of one seed
restart_seeds = list(range(1, 1001))

# 26,000 fits take ~40 min on 12 threads, so results are cached next to the
# notebook; set RERUN_RESTARTS = True after changing the seeds or tolerances
RERUN_RESTARTS = False
RESTART_CACHE = NOTEBOOK_DIR / "restart_rel_error_results.csv.gz"
restart_model = cmdstanpy.CmdStanModel(stan_file=STAN_FILE)


def restart_rel_error(tol, gs, seed):
    out_dir = tempfile.mkdtemp()  # per-run dir: parallel runs would collide
    try:
        fit = restart_model.variational(
            data=stan_data, tol_rel_obj=tol, seed=seed, grad_samples=gs,
            draws=50_000, require_converged=False, iter=100_000, eval_elbo=10,
            output_dir=out_dir,
        )
    except RuntimeError:
        # e.g. "gradient calculation failed" -- real ADVI instability
        return dict(tol=tol, grad_samples=gs, seed=seed, rel_error=np.nan)
    finally:
        shutil.rmtree(out_dir, ignore_errors=True)
    std_hat = fit.variational_sample[:, fit.column_names.index("sigma_sq")].std()
    return dict(tol=tol, grad_samples=gs, seed=seed, rel_error=abs(best_std - std_hat) / best_std)


if RESTART_CACHE.exists() and not RERUN_RESTARTS:
    restart_df = pd.read_csv(RESTART_CACHE)
else:
    cmdstanpy.disable_logging()
    with ThreadPoolExecutor(max_workers=max(1, os.cpu_count() - 2)) as pool:
        restart_df = pd.DataFrame(pool.map(
            lambda args: restart_rel_error(*args),
            [(tol, gs, seed) for tol in tols for gs in (1, 100) for seed in restart_seeds],
        ))
    cmdstanpy.enable_logging()
    restart_df.to_csv(RESTART_CACHE, index=False)

print(f"{len(restart_df)} runs, {restart_df['rel_error'].isna().sum()} numerical failures")
restart_summary = (
    restart_df.groupby(["grad_samples", "tol"])["rel_error"]
    .agg(["mean", "std", "count"]).reset_index()
)
restart_summary

26000 runs, 13 numerical failures


,grad_samples,tol,mean,std,count
0,1,0.000001,0.033799,0.023807,999
1,1,0.000100,0.033947,0.024144,999
2,1,0.000500,0.034702,0.028054,999
3,1,0.000600,0.035463,0.031911,999
4,1,0.000700,0.035992,0.033098,999
5,1,0.000750,0.036431,0.035819,999
6,1,0.000800,0.038064,0.040055,999
7,1,0.000900,0.046098,0.050700,999
8,1,0.001000,0.056881,0.062096,999
9,1,0.005000,0.188028,0.189179,999


In [7]:
restart_colors = {1: ("blue", "rgba(0, 0, 255, 0.15)"), 100: ("purple", "rgba(128, 0, 128, 0.15)")}
restart_names = {1: "1 MC sample (default)", 100: "100 MC samples (adjusted)"}
y_max = 1

fig = go.Figure()
for gs, (line_color, band_color) in restart_colors.items():
    s = restart_summary[restart_summary["grad_samples"] == gs].sort_values("tol")
    # shaded band = mean +/- 1 std across the 100 restarts: draw the upper
    # edge, then fill down to the lower edge
    fig.add_trace(go.Scatter(
        x=s["tol"], y=s["mean"] + s["std"], mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=s["tol"], y=s["mean"] - s["std"], mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor=band_color,
        name=f"{restart_names[gs]}: mean ± 1 std. dev.", hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=s["tol"], y=s["mean"], mode="lines+markers",
        line=dict(color=line_color), marker=dict(color=line_color),
        name=f"{restart_names[gs]}: mean", customdata=s["std"],
        hovertemplate="tol=%{x:.1e}<br>mean=%{y:.3f}<br>std=%{customdata:.3f}<extra></extra>",
    ))
    # the y-axis is fixed to [0, 1], so means above 1 would otherwise silently
    # vanish -- pin them to the top edge with an up-arrow and their true value
    for tol, m in zip(s["tol"], s["mean"]):
        if m > y_max:
            # annotation x on a log axis is given in log10 units
            # stack the two settings' labels so they don't overprint
            fig.add_annotation(
                x=np.log10(tol), y=y_max, xref="x", yref="y", yanchor="top",
                yshift=0 if gs == 1 else -34,
                text=f"▲<br>{m:.2f}", showarrow=False, font=dict(size=11, color=line_color),
            )

fig.add_vline(x=0.01, line=dict(color="black", dash="dash"))
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="lines", line=dict(color="black", dash="dash"),
    name="default tolerance",
))

label_tols = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]
fig.update_xaxes(
    title_text="Convergence tolerance (tol_rel_obj)", type="log",
    tickmode="array", tickvals=label_tols,
    ticktext=[f"{t:.0e}" for t in label_tols], tickangle=0,
)
fig.update_yaxes(title_text="Magnitude of relative error", type="linear", range=[0, y_max])
fig.update_layout(
    title=dict(
        text=(
            "Relative error of output var. approx. post. std. dev to the best "
            "var approx post. std. dev.<br>vs convergence tolerance parameter, "
            f"over {len(restart_seeds)} random seed restarts"
        ),
        subtitle=dict(text=(
            r"$\text{line = mean, shaded = mean} \pm 1 \text{ std. dev. across restarts of } "
            r"\frac{|(\sigma_p)_{\text{best var approx}} - \hat{\sigma}_p|}"
            r"{(\sigma_p)_{\text{best var approx}}}$"
        )),
    ),
    # filled traces flip plotly's default legend order; keep 1 MC first
    legend=dict(traceorder="normal"),
    margin=dict(t=170),
    height=620, width=1100,
)
show_with_mathjax(fig)

## When does a loose tolerance let ADVI stop early, and how early?

The flat region on the left of the plot above exists because, for small `tol_rel_obj`, Stan's ADVI never satisfies its own convergence check within the `iter=100_000` budget -- it just runs out the clock. Looser tolerances let it declare convergence before the budget is spent.

Here we map that out across many datasets and settings, asking (a) which tolerances let ADVI stop before the iteration budget, and (b) at what iteration it stops when it does. We sweep:

- sample size `n` = 3, 10, 30, 100, 300, 500, 1000
- the InvGamma prior: (`alpha_prior`, `beta_prior`) = (1, 1), (4, 3), (20, 15) -- `alpha_prior` is the shape
- the likelihood's standard deviation `sigma_like` = 0.5, 2.0
- 10 simulated datasets per setting, and 10 ADVI seeds on each dataset
- 11 tolerances from `1e-6` to `1`
- `grad_samples` = 1 (default) and 100

That is 92,400 ADVI runs. The results are cached in `tolerance_sweep_results.csv.gz` next to the Stan model; set `RERUN_SWEEP = True` to regenerate them (several hours on 12 threads; an interrupted run resumes from its checkpoint file).

In [8]:
import itertools
import shutil
import tempfile
from concurrent.futures import ThreadPoolExecutor

ITER_CAP = 100_000

sweep_n_samples = [3, 10, 30, 100, 300, 500, 1000]
sweep_priors = [(1.0, 1.0), (4.0, 3.0), (20.0, 15.0)]  # (alpha_prior = InvGamma shape, beta_prior)
sweep_sigma_likes = [0.5, 2.0]
sweep_tols = [1e-6, 1e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0]
sweep_data_seeds = list(range(1, 11))  # which simulated dataset
sweep_advi_seeds = list(range(1, 11))  # which ADVI run on that dataset
sweep_grad_samples = [1, 100]

# a full re-run is ~92k ADVI fits (several hours on 12 threads), so results are
# cached next to the notebook; set RERUN_SWEEP = True after changing the grid.
# while running, finished chunks are appended to SWEEP_PARTIAL so an
# interrupted sweep resumes where it left off instead of starting over
RERUN_SWEEP = False
SWEEP_CACHE = NOTEBOOK_DIR / "tolerance_sweep_results.csv.gz"
SWEEP_PARTIAL = NOTEBOOK_DIR / "tolerance_sweep_results.partial.csv"
SWEEP_KEY = ["n_samples", "alpha_prior", "beta_prior", "sigma_like",
             "data_seed", "advi_seed", "tol", "grad_samples"]

sweep_model = cmdstanpy.CmdStanModel(stan_file=STAN_FILE)


def run_one(n, prior, sigma_like_, data_seed, advi_seed, tol, gs):
    """
    One ADVI fit. Returns the iteration at which Stan declared convergence
    (stop_iter < ITER_CAP => stopped early), or stop_iter = ITER_CAP if it
    ran out the iteration budget. A numerical failure is recorded as failed.
    """
    alpha_p, beta_p = prior
    y = gaussian_1d(n, mu_like, sigma_like=sigma_like_, seed=data_seed).astype(float)
    row = dict(n_samples=n, alpha_prior=alpha_p, beta_prior=beta_p, sigma_like=sigma_like_,
               data_seed=data_seed, advi_seed=advi_seed, tol=tol, grad_samples=gs)
    # separate output dir per run: cmdstanpy names files by timestamp, which
    # collides across parallel threads
    out_dir = tempfile.mkdtemp()
    try:
        fit = sweep_model.variational(
            data={"N": n, "y": y, "mu_like": mu_like,
                  "alpha_prior": alpha_p, "beta_prior": beta_p},
            tol_rel_obj=tol, seed=advi_seed, grad_samples=gs, draws=10,
            require_converged=False, iter=ITER_CAP, eval_elbo=10,
            output_dir=out_dir,
        )
        console = open(fit.runset.stdout_files[0]).read()
    except RuntimeError:
        # e.g. "gradient calculation failed" -- real ADVI instability
        return {**row, "failed": True, "stop_iter": np.nan}
    finally:
        shutil.rmtree(out_dir, ignore_errors=True)

    stop_iter = ITER_CAP
    for line in console.splitlines():
        if "ELBO CONVERGED" in line:
            stop_iter = int(line.split()[0])
            break
    return {**row, "failed": False, "stop_iter": stop_iter}


def run_sweep(chunk_size=2000):
    grid = list(itertools.product(
        sweep_n_samples, sweep_priors, sweep_sigma_likes,
        sweep_data_seeds, sweep_advi_seeds, sweep_tols, sweep_grad_samples,
    ))
    done = set()
    if SWEEP_PARTIAL.exists():
        partial = pd.read_csv(SWEEP_PARTIAL)
        done = set(partial[SWEEP_KEY].itertuples(index=False, name=None))
    todo = [
        args for args in grid
        if (args[0], *args[1], args[2], args[3], args[4], args[5], args[6]) not in done
    ]
    print(f"{len(grid):,} runs in grid, {len(grid) - len(todo):,} already done")

    # cmdstanpy logs a few lines per fit -- silence them for the sweep
    cmdstanpy.disable_logging()
    try:
        with ThreadPoolExecutor(max_workers=max(1, os.cpu_count() - 2)) as pool:
            for start in range(0, len(todo), chunk_size):
                chunk = todo[start:start + chunk_size]
                rows = pd.DataFrame(pool.map(lambda args: run_one(*args), chunk))
                rows.to_csv(SWEEP_PARTIAL, mode="a", index=False, header=not SWEEP_PARTIAL.exists())
                print(f"  {len(grid) - len(todo) + start + len(chunk):,} / {len(grid):,} done", flush=True)
    finally:
        cmdstanpy.enable_logging()

    # keep only rows in the current grid (the checkpoint may hold runs from
    # an earlier, larger grid)
    sweep_df = pd.read_csv(SWEEP_PARTIAL)
    grid_keys = {(a[0], *a[1], a[2], a[3], a[4], a[5], a[6]) for a in grid}
    in_grid = [k in grid_keys for k in sweep_df[SWEEP_KEY].itertuples(index=False, name=None)]
    sweep_df = sweep_df[in_grid].reset_index(drop=True)
    assert len(sweep_df) == len(grid), f"{len(sweep_df)} rows for a {len(grid)}-run grid"
    sweep_df.to_csv(SWEEP_CACHE, index=False)
    SWEEP_PARTIAL.unlink()
    return sweep_df


if SWEEP_CACHE.exists() and not RERUN_SWEEP:
    sweep_df = pd.read_csv(SWEEP_CACHE)
else:
    sweep_df = run_sweep()

sweep_df["stopped_early"] = (~sweep_df["failed"]) & (sweep_df["stop_iter"] < ITER_CAP)
print(f"{len(sweep_df):,} runs, {sweep_df['failed'].sum():,} numerical failures")
sweep_df.groupby(["grad_samples", "tol"])["stopped_early"].mean().unstack(0)

92,400 runs, 601 numerical failures


grad_samples,1,100
tol,,
0.000001,0.002619,0.000238
0.000010,0.003333,0.000238
0.000100,0.169762,0.276667
0.000300,0.407619,0.493810
0.001000,0.600952,0.648810
0.003000,0.749762,0.790238
0.010000,0.896667,0.925714
0.030000,0.987143,0.998810
0.100000,0.987381,0.998810


**How to read the plot below.** Top row: 1 MC sample (Stan's default); bottom row: 100 MC samples. Left column (a): each thin line is one prior/likelihood config, giving the % of its 100 runs (10 datasets x 10 ADVI seeds) that stopped before the iteration budget at each tolerance; lines are colored by sample size. Right column (b): for the runs that *did* stop early, the median stopping iteration (marker) and 10th-90th percentile range (bar), per sample size.

**Takeaways.**

- **(a) Which tolerances risk early convergence depends mostly on the amount of data.** With 1 MC sample, the `n=1000` configs already stop early in 74% of runs at `tol_rel_obj=1e-4`, and the `n=500` configs in every run from `3e-4` up, while the `n=3` configs stop early in only 36% of runs at `1e-2` (and in all runs from `3e-2` up). The prior's shape barely moves the curves; the larger `sigma_like` shifts them toward tighter tolerances (e.g. at `1e-4`, 26-27% of 1-MC runs stop early with `sigma_like=2` vs. 8-9% with `sigma_like=0.5`).
- **At Stan's default `tol_rel_obj=0.01`, early stopping is the norm**, not the exception: every run with `n >= 10` stopped early, with either grad_samples setting.
- **Even a very tight tolerance isn't a guarantee.** 27 of the 16,684 runs at `tol_rel_obj <= 1e-5` (0.16%) still stopped early -- 19 of them at iteration 30, where the noisy ELBO estimate happened to barely change between the first few convergence checks.
- **(b) When it stops early, it usually stops *very* early.** Near the tolerance where early stopping sets in, the stopping iteration is widely spread (from tens to tens of thousands of iterations). Once the tolerance is `0.1` or looser, the median run stops at iteration 20-30 -- after 2-3 ELBO evaluations, out of the 10,000 the budget allows -- and 90% of runs have stopped by iteration 50. At the default `0.01` the median stopping iteration is between 30 and 260 for every `n`.
- **100 MC samples makes early stopping *more* likely at a given tolerance, not less** (e.g. at `1e-4`: 43% vs. 6% of runs for `n=300`, 100% vs. 74% for `n=1000`) -- presumably because a less noisy ELBO estimate satisfies the relative-change check sooner.

Note that "stopped early" here just means "stopped before the iteration budget"; whether stopping early hurt the approximation is what the first plot in this notebook measures, for one config.

In [9]:
from plotly.colors import sample_colorscale
from plotly.subplots import make_subplots

CONFIG_COLS = ["n_samples", "alpha_prior", "beta_prior", "sigma_like"]
ok_df = sweep_df[~sweep_df["failed"]]
runs_per_config = len(sweep_data_seeds) * len(sweep_advi_seeds)

# tolerances are half-decade spaced, so plotting against their index is
# evenly spaced in log(tol) and leaves room to offset the n groups in (b)
tol_idx = {t: i for i, t in enumerate(sweep_tols)}
default_tol_idx = tol_idx[1e-2]

# sequential colors by sample size (the factor that moves the curves most);
# stop short of viridis' pale yellow end so every line stays visible
n_colors = dict(zip(sweep_n_samples, sample_colorscale("Viridis", np.linspace(0, 0.88, len(sweep_n_samples)))))
n_offsets = dict(zip(sweep_n_samples, np.linspace(-0.33, 0.33, len(sweep_n_samples))))
gs_names = {1: "1 MC sample (default)", 100: "100 MC samples"}

# rows = grad_samples setting, columns = (a) how often / (b) how early
fig = make_subplots(
    rows=2, cols=2, shared_xaxes=True,
    vertical_spacing=0.12, horizontal_spacing=0.08,
    subplot_titles=[
        f"{gs_names[gs]}: {panel}"
        for gs in sweep_grad_samples
        for panel in ("(a) % of runs that stopped early", "(b) iteration at which they stopped")
    ],
)

for row, gs in enumerate(sweep_grad_samples, start=1):
    gs_df = ok_df[ok_df["grad_samples"] == gs]

    # (a) one thin line per prior/likelihood config, colored by n
    frac = gs_df.groupby(CONFIG_COLS + ["tol"])["stopped_early"].mean().reset_index()
    for (n, a, b, s), cfg_df in frac.groupby(CONFIG_COLS):
        cfg_df = cfg_df.sort_values("tol")
        fig.add_trace(go.Scatter(
            x=cfg_df["tol"].map(tol_idx), y=100 * cfg_df["stopped_early"],
            mode="lines+markers", line=dict(color=n_colors[n], width=1.2),
            marker=dict(size=4), opacity=0.7,
            name=f"n = {n}", legendgroup=f"n{n}",
            showlegend=(row == 1 and a == 4.0 and s == 0.5),
            customdata=cfg_df["tol"],
            hovertemplate=(
                f"{gs_names[gs]}<br>n={n}, alpha={a:g}, beta={b:g}, sigma_like={s:g}"
                "<br>tol=%{customdata:.0e}: %{y:.0f}% of runs stopped early<extra></extra>"
            ),
        ), row=row, col=1)

    # (b) among runs that DID stop early, the median stopping iteration
    # (marker) and its 10th-90th percentile range (bar), per n
    early_df = gs_df[gs_df["stopped_early"]]
    for n in sweep_n_samples:
        q = (
            early_df[early_df["n_samples"] == n].groupby("tol")["stop_iter"]
            .agg(med="median", lo=lambda v: v.quantile(0.1), hi=lambda v: v.quantile(0.9), count="size")
            .reset_index()
        )
        n_runs = len(sweep_priors) * len(sweep_sigma_likes) * runs_per_config
        fig.add_trace(go.Scatter(
            x=q["tol"].map(tol_idx) + n_offsets[n], y=q["med"], mode="markers",
            marker=dict(color=n_colors[n], size=6),
            error_y=dict(type="data", symmetric=False, array=q["hi"] - q["med"],
                         arrayminus=q["med"] - q["lo"], color=n_colors[n], thickness=1.5, width=0),
            name=f"n = {n}", legendgroup=f"n{n}", showlegend=False,
            customdata=np.stack([q["tol"], q["lo"], q["hi"], q["count"]], axis=-1),
            hovertemplate=(
                f"{gs_names[gs]}<br>n={n}, tol=%{{customdata[0]:.0e}}<br>median stop: iteration %{{y:,.0f}}"
                "<br>10th-90th pct: %{customdata[1]:,.0f} - %{customdata[2]:,.0f}"
                f"<br>%{{customdata[3]:,}} of {n_runs:,} runs stopped early<extra></extra>"
            ),
        ), row=row, col=2)

    fig.add_hline(y=ITER_CAP, line=dict(color="grey", dash="dot"), row=row, col=2)

fig.add_vline(x=default_tol_idx, line=dict(color="black", dash="dash"), row="all", col="all")
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="lines", line=dict(color="black", dash="dash"),
    name="Stan's default tolerance (1e-2)",
))
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="lines", line=dict(color="grey", dash="dot"),
    name=f"iteration budget (iter={ITER_CAP:,})",
))

fig.update_xaxes(
    tickmode="array", tickvals=list(tol_idx.values()),
    ticktext=[f"{t:g}" if t >= 0.1 else f"{t:.0e}".replace("e-0", "e-") for t in sweep_tols],
    tickangle=0, range=[-0.6, len(sweep_tols) - 0.4], showgrid=False, showticklabels=True,
)
fig.update_xaxes(title_text="Convergence tolerance (tol_rel_obj)", row=2)
fig.update_yaxes(range=[-3, 103], title_text="% of runs that stopped<br>before the iteration budget", col=1)
fig.update_yaxes(type="log", range=[1, 5.25], dtick=1, title_text="Iteration at which Stan<br>declared convergence", col=2)

n_configs = len(sweep_n_samples) * len(sweep_priors) * len(sweep_sigma_likes)
fig.update_layout(
    title=dict(
        text=(
            "(a) Which tolerances let ADVI stop early, and (b) how early it stops"
            f"<br><span style='font-size:13px;color:grey'>{len(sweep_df):,} ADVI runs = "
            f"{n_configs} prior/likelihood configs x {len(sweep_data_seeds)} datasets x {len(sweep_advi_seeds)} ADVI seeds "
            f"x {len(sweep_tols)} tolerances x 2 grad_samples settings"
            f"<br>(a): one line per config = % of its {runs_per_config} runs that stopped before the budget. "
            "(b): median (marker) and 10th-90th percentile (bar) stopping iteration of those runs, per n.</span>"
        ),
    ),
    legend=dict(title_text="Sample size", tracegroupgap=4),
    height=860, width=1300, margin=dict(t=175, b=80),
)
fig.show()